# Feature Engineering

## Objective

The objective of this notebook is to prepare the cleaned dataset for machine learning by:

- Separating features and target
- Identifying numerical and categorical variables
- Splitting the dataset
- Building preprocessing pipelines
- Encoding categorical variables
- Scaling numerical variables
- Producing machine-learning-ready data

In [1]:
# Libraries
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Load Dataset
df = pd.read_csv("../data/cleaned.csv")

df.head()

,Gender,Age,Ethnicity,T_atm,Humidity,Distance,T_offset1,Max1R13_1,Max1L13_1,aveAllR13_1,...,T_FHRC1,T_FHLC1,T_FHBC1,T_FHTC1,T_FH_Max1,T_FHC_Max1,T_Max1,T_OR1,T_OR_Max1,aveOralF
0,Male,41-50,White,24.0,28.0,0.8,0.7025,35.0300,35.3775,34.4000,...,33.4775,33.3725,33.4925,33.0025,34.5300,34.0075,35.6925,35.6350,35.6525,36.85
1,Female,31-40,Black or African-American,24.0,26.0,0.8,0.7800,34.5500,34.5200,33.9300,...,34.0550,33.6775,33.9700,34.0025,34.6825,34.6600,35.1750,35.0925,35.1075,37.00
2,Female,21-30,White,24.0,26.0,0.8,0.8625,35.6525,35.5175,34.2775,...,34.8275,34.6475,34.8200,34.6700,35.3450,35.2225,35.9125,35.8600,35.8850,37.20
3,Female,21-30,Black or African-American,24.0,27.0,0.8,0.9300,35.2225,35.6125,34.3850,...,34.4225,34.6550,34.3025,34.9175,35.6025,35.3150,35.7200,34.9650,34.9825,36.85
4,Male,18-20,White,24.0,27.0,0.8,0.8950,35.5450,35.6650,34.9100,...,35.1600,34.3975,34.6700,33.8275,35.4175,35.3725,35.8950,35.5875,35.6175,36.80


In [3]:
# Separate features and target
TARGET = "aveOralF"

X = df.drop(columns=["aveOralF"])

y = df[TARGET]

In [4]:
# Identify Feature Types
categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

In [5]:
print("Categorical Features")
print(categorical_features)

print()

print("Numerical Features")
print(numerical_features)

Categorical Features
['Gender', 'Age', 'Ethnicity']

Numerical Features
['T_atm', 'Humidity', 'Distance', 'T_offset1', 'Max1R13_1', 'Max1L13_1', 'aveAllR13_1', 'aveAllL13_1', 'T_RC1', 'T_RC_Dry1', 'T_RC_Wet1', 'T_RC_Max1', 'T_LC1', 'T_LC_Dry1', 'T_LC_Wet1', 'T_LC_Max1', 'RCC1', 'LCC1', 'canthiMax1', 'canthi4Max1', 'T_FHCC1', 'T_FHRC1', 'T_FHLC1', 'T_FHBC1', 'T_FHTC1', 'T_FH_Max1', 'T_FHC_Max1', 'T_Max1', 'T_OR1', 'T_OR_Max1']


In [6]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [7]:
print(X_train.shape)
print(X_test.shape)

(816, 33)
(204, 33)


In [8]:
# Numerical Pipeline
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),

        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [9]:
# Categorical Pipeline
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),

        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [10]:
# Combine Pipelines
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numerical_features
        ),

        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [11]:
# Fit the Preprocessor
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [12]:
# Verify Shapes
print("Training Features")

print(X_train_processed.shape)

print()

print("Testing Features")

print(X_test_processed.shape)

Training Features
(816, 46)

Testing Features
(204, 46)


In [13]:
# Save the Preprocessor

joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

['../models/preprocessor.pkl']

## Feature Engineering Summary

The following preprocessing steps were completed:

- Selected `aveOralF` as the prediction target.
- Removed `aveOralM` to prevent target leakage.
- Identified categorical and numerical features.
- Split the dataset into training and testing sets.
- Built preprocessing pipelines for numerical and categorical variables.
- Applied median imputation and feature scaling to numerical features.
- Applied one-hot encoding to categorical features.
- Combined preprocessing steps using a `ColumnTransformer`.
- Saved the fitted preprocessing pipeline for reuse during model deployment.